# CrashLens End-to-End Test

This notebook demonstrates CrashLens tracking real GPU workloads with actual CUDA/PyTorch.

**Requirements:**
- GPU runtime (Runtime → Change runtime type → GPU)
- CrashLens backend deployed (Railway, etc.)
- API key (if required by your backend)

## 1. Setup

In [ ]:
# Install CrashLens SDK from GitHub
!pip install -q torch torchvision
!pip install -q git+https://github.com/kavinsaravan/ML-Failure-Doctor.git#subdirectory=crashlens-sdk

# Install GPU metrics support
!pip install -q nvidia-ml-py

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Configure your backend URL and API key
BACKEND_URL = "https://your-backend.railway.app"  # Replace with your backend URL
API_KEY = os.getenv("CRASHLENS_API_KEY", None)  # Optional: set if backend requires auth

# Verify GPU availability
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Initialize CrashLens Tracker

In [ ]:
# Initialize CrashLens SDK
from crashlens import WorkloadTracker

tracker = WorkloadTracker(BACKEND_URL, api_key=API_KEY)
print("✅ Tracker initialized with GPU metrics support")

## 3. Test 1: Successful Training

In [ ]:
# Simple CNN model
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(16 * 13 * 13, 10)
    
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = x.view(-1, 16 * 13 * 13)
        return self.fc1(x)

# Track the training with SDK (captures GPU metrics automatically)
with tracker.track("E2E Test - Successful Training"):
    # Create dummy data
    X = torch.randn(100, 3, 28, 28)
    y = torch.randint(0, 10, (100,))
    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset, batch_size=32)
    
    # Train
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SimpleCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters())
    
    print(f"Training on {device}")
    
    for epoch in range(3):
        for batch_X, batch_y in loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
        
        print(f"Epoch {epoch+1}/3 - Loss: {loss.item():.4f}")

print("✅ Training completed")

## 4. Test 2: GPU Out of Memory (Intentional Failure)

In [ ]:
# OOM test with SDK (captures GPU metrics showing memory climb to OOM)
try:
    with tracker.track("E2E Test - GPU OOM"):
        # Only run on CUDA, skip on CPU
        if not torch.cuda.is_available():
            print("⏭️  Skipping OOM test (no GPU)")
        else:
            print("Attempting to allocate large tensor...")
            
            # Allocate ~40GB (exceeds T4's 15GB)
            # GPU metrics will show memory climbing to ~15GB before OOM
            huge_tensor = torch.randn(100000, 100000, device="cuda")
            
            print("Allocation succeeded (unexpected!)")
    
except RuntimeError as e:
    print(f"❌ Failed as expected: {str(e)[:100]}...")
    print("\nGPU metrics captured the memory climb to OOM!")
    print("Check the dashboard to see memory usage approaching 15GB")

## 5. Test 3: Dependency Error

In [ ]:
try:
    with tracker.track("E2E Test - Dependency Error"):
        # Try to import a non-existent package
        print("Importing dependencies...")
        import nonexistent_package_12345
    
except ModuleNotFoundError as e:
    print(f"❌ Failed as expected: ModuleNotFoundError")

## 6. Verify Results in Dashboard

Visit your CrashLens dashboard to see:
- All 3 workloads tracked
- Real GPU metrics from CUDA
- AI diagnosis for failed workloads
- Wasted GPU time calculations

**Dashboard URL:** Replace with your deployed frontend URL

In [ ]:
# List all workloads
import requests
resp = requests.get(f"{BACKEND_URL}/workloads")
workloads = resp.json()

print("\n📊 Recent Workloads:")
for w in workloads[-5:]:  # Show last 5
    status_emoji = "✅" if w["status"] == "succeeded" else "❌"
    print(f"{status_emoji} [{w['id']}] {w['name']} - {w['status']}")

print(f"\n🔗 View dashboard with GPU metrics: {BACKEND_URL.replace(':8080', ':3000')}")
print("\n💡 The OOM workload should show GPU memory climbing to ~15GB in the metrics chart!")